# Exercícios — Aula 14: Processamento Massivo

Cenário: uma concessionária de rodovias registra cada passagem de veículo pelos seus pórticos de pedágio eletrônico — placa, pórtico, instante e velocidade medida. A célula abaixo gera um registro sintético de um dia e o de outro dia, e fornece uma implementação de HyperLogLog para os exercícios 5 e 8.

Execute a célula antes de começar.

In [ ]:
import hashlib
import math
import random

import numpy as np
import matplotlib.pyplot as plt

random.seed(2026)
gerador = np.random.default_rng(2026)

letras = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def placa_aleatoria():
    # formato Mercosul: ABC1D23
    return random.choice(letras) + random.choice(letras) + random.choice(letras) + str(random.randint(0, 9)) + random.choice(letras) + str(random.randint(0, 9)) + str(random.randint(0, 9))

frota = set()
while len(frota) < 80_000:
    frota.add(placa_aleatoria())
frota = sorted(frota)
random.shuffle(frota)

porticos = ["P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08"]

def gerar_dia(frota_do_dia, n_passagens):
    pesos = 1 / np.arange(1, len(frota_do_dia) + 1) ** 0.9
    pesos = pesos / pesos.sum()
    indices = gerador.choice(len(frota_do_dia), size=n_passagens, p=pesos)
    registro = []
    for i in indices:
        placa = frota_do_dia[i]
        portico = random.choice(porticos)
        instante = random.randint(0, 86_399)
        velocidade = round(random.gauss(95, 12), 1)
        registro.append((placa, portico, instante, velocidade))
    return registro

dia1 = gerar_dia(frota[:60_000], 300_000)
dia2 = gerar_dia(frota[20_000:80_000], 300_000)
print(len(dia1), "passagens no dia 1;", len(dia2), "no dia 2")
print("exemplo:", dia1[0])

# HyperLogLog fornecido (exercícios 5 e 8)
def hll_novo(p):
    return [0] * (2 ** p)

def hll_inserir(registradores, item):
    p = int(math.log2(len(registradores)))
    h = int.from_bytes(hashlib.blake2b(item.encode(), digest_size=8).digest(), "big")
    j = h >> (64 - p)
    resto = h & ((1 << (64 - p)) - 1)
    rho = (64 - p) - resto.bit_length() + 1
    if rho > registradores[j]:
        registradores[j] = rho

def hll_estimar(registradores):
    m = len(registradores)
    alfa = 0.7213 / (1 + 1.079 / m)
    soma = 0.0
    zerados = 0
    for valor in registradores:
        soma = soma + 2.0 ** (-valor)
        if valor == 0:
            zerados = zerados + 1
    estimativa = alfa * m * m / soma
    if estimativa < 2.5 * m and zerados > 0:
        estimativa = m * math.log(m / zerados)
    return estimativa

### 1. Escolha da estrutura

Para cada necessidade abaixo, indique a estrutura probabilística adequada entre filtro de Bloom, filtro de cuco, *count-min sketch*, *top-k*, HyperLogLog, MinHash e *t-digest*. Justifique pela pergunta respondida e pela **direção do erro**: diga qual erro a estrutura comete e por que esse erro é tolerável para a decisão descrita.

a) Antes de consultar o cadastro nacional, o pórtico verifica se a placa está na lista de veículos com pendência de pagamento; a lista é atualizada uma vez por mês.

b) A mesma lista do item (a), agora atualizada a cada pagamento: a placa sai da lista quando o débito é quitado.

c) O painel da operação mostra, a cada hora, as 20 placas com mais passagens.

d) O relatório mensal à agência reguladora informa quantos veículos distintos usaram a rodovia no mês, a partir dos relatórios diários.

e) A concessionária quer saber o quanto a frota que usa a rodovia num dia útil se parece com a de um domingo, comparando os dois dias.

f) O contrato de concessão exige que o percentil 95 do tempo de espera nas cabines manuais fique abaixo de 3 minutos.

g) O sistema antifraude sinaliza placas com mais de 30 passagens por dia, padrão típico de placa clonada.

*Respostas:*

### 2. Dimensionamento

Calcule, mostrando as fórmulas usadas:

a) $m$ (em bits e em MB) e $k$ de um filtro de Bloom para $10^7$ placas com taxa de falso positivo de 0,1%. Compare com a memória de guardar as placas (7 bytes cada) num conjunto, sem considerar a sobrecarga da tabela *hash*.

b) $w$, $d$ e a memória (contadores de 32 bits) de um *count-min* com $\varepsilon = 10^{-5}$ e $\delta = 0{,}001$. Com $N = 3 \cdot 10^6$ passagens por dia, qual o maior erro que a estimativa de uma placa pode ter com probabilidade de pelo menos 99,9%? Esse erro permite aplicar com segurança o limiar de 30 passagens do item 1g?

c) O menor $p$ de um HyperLogLog cujo erro padrão relativo fique abaixo de 1%, e a memória com registradores de 6 bits.

d) O número $k$ de funções de *hash* do MinHash para que o erro padrão da estimativa de Jaccard fique abaixo de 0,02, qualquer que seja $J$.

In [ ]:
# 2. Dimensionamento

### 3. Filtro de Bloom

Implemente um filtro de Bloom dimensionado para as placas distintas do dia 1 com taxa de falso positivo de 1%. Use `hashlib.blake2b` e *hash* duplo para obter as $k$ posições.

a) Insira as placas distintas do dia 1.

b) Verifique que nenhuma placa inserida responde "ausente".

c) Meça a taxa de falso positivo consultando as placas da `frota` que não passaram no dia 1 e compare com a taxa prevista pela fórmula.

d) Repita a medição com metade dos bits ($m/2$, com o $k$ ótimo recalculado) e explique a diferença.

In [ ]:
# 3. Filtro de Bloom

def bloom_posicoes(placa, m, k):
    # devolve a lista das k posições da placa no vetor de m bits
    pass

def bloom_inserir(bits, placa, m, k):
    pass

def bloom_consultar(bits, placa, m, k):
    # True = provavelmente presente; False = certamente ausente
    pass

### 4. *Count-min sketch*

Implemente um *count-min* com $\varepsilon = 0{,}0005$ e $\delta = 0{,}01$ e conte as passagens por placa do dia 1.

a) Verifique que nenhuma estimativa fica abaixo da contagem exata (calculada com um dicionário).

b) Calcule a fração de placas cuja estimativa passa de $f(x) + \varepsilon N$ e compare com $\delta$.

c) Monte uma tabela com a contagem exata, a estimativa e o erro relativo para as placas nas posições 1, 10, 100, 1.000 e 10.000 do ranking de passagens. Explique, a partir da fórmula do erro, por que o erro relativo cresce com a posição.

In [ ]:
# 4. Count-min sketch

### 5. HyperLogLog

Use o HyperLogLog fornecido na primeira célula.

a) Estime o número de placas distintas do dia 1 com $p = 8, 10, 12$ e $14$ e compare cada estimativa com `len(set(...))`. Compare o erro observado com $1{,}04/\sqrt{m}$.

b) Construa um *sketch* para cada dia, com $p = 14$, e funda os dois pelo máximo registrador a registrador. Compare a estimativa da fusão com o número exato de placas distintas nos dois dias juntos e com a soma das contagens distintas de cada dia. Explique por que a soma está errada e a fusão não.

In [ ]:
# 5. HyperLogLog

### 6. MinHash

Estime a similaridade de Jaccard entre o conjunto de placas do dia 1 e o do dia 2.

a) Calcule o valor exato de $J$.

b) Implemente o MinHash com $k$ funções de *hash* (por exemplo, `blake2b` com um `salt` diferente para cada função) e estime $J$ com $k = 20, 50, 100$ e $200$.

c) Compare o erro de cada estimativa com $\sqrt{J(1-J)/k}$.

In [ ]:
# 6. MinHash

### 7. MapReduce

Divida o registro do dia 1 em 6 blocos e simule o MapReduce com funções `fase_map`, `embaralhar` (particionamento por `zlib.crc32(chave) % R`, com $R = 3$) e `fase_reduce`.

a) Escreva o *map* e o *reduce* do número de passagens por pórtico. Confira o resultado contra uma contagem direta.

b) Acrescente um combinador ao item (a) e compare o número de pares embaralhados com e sem combinador.

c) Escreva o *map*, o combinador e o *reduce* da velocidade média por pórtico. O combinador deve produzir o mesmo resultado que o cálculo direto mesmo com blocos de tamanhos diferentes: mostre isso dividindo o registro em blocos desiguais e comparando com a média das médias locais.

In [ ]:
# 7. MapReduce

### 8. MapReduce com HyperLogLog

Escreva o *map* e o *reduce* do número de placas distintas por pórtico, com o HyperLogLog ($p = 12$) como combinador e como redutor: cada mapeador emite um *sketch* por pórtico, e o redutor funde os *sketches* de cada pórtico.

a) Compare as estimativas com a contagem exata por pórtico.

b) Compare o número de *sketches* embaralhados com o número de pares (pórtico, placa) que a versão exata embaralharia.

c) Explique por que somar as contagens distintas de cada bloco daria um resultado errado.

In [ ]:
# 8. MapReduce com HyperLogLog

*Conclusões:*